# Backend 教程：硬件信息、拓扑、排序与导出

本教程覆盖：
- 创建 `Backend`（Simulator / 真实芯片 / 自定义）
- 读取拓扑图信息（节点、边、门基）
- 边保真度阈值过滤 `edge_filtered_graph`
- 导出芯片拓扑图 `draw(...)`
- 获取芯片信息 `get_chip_info(...)`

In [1]:
from pathlib import Path

from fieldqkit.api.backend import Backend, list_available_hardware

def section(title: str):
    print("\n" + "=" * 18, title, "=" * 18)

## 1) 创建后端并检查拓扑

In [5]:
section("backend")
list_available_hardware("quafu")


================== backend ==================


[{'provider': 'quafu',
  'hardware_name': 'Baihua',
  'queue_length': 1018,
  'status': None,
  'is_toll': None,
  'raw': {'queue_length': 1018}},
 {'provider': 'quafu',
  'hardware_name': 'Baiwang',
  'queue_length': 12,
  'status': None,
  'is_toll': None,
  'raw': {'queue_length': 12}},
 {'provider': 'quafu',
  'hardware_name': 'Dongling',
  'queue_length': 0,
  'status': None,
  'is_toll': None,
  'raw': {'queue_length': 0}},
 {'provider': 'quafu',
  'hardware_name': 'Fenghuang',
  'queue_length': 0,
  'status': None,
  'is_toll': None,
  'raw': {'queue_length': 0}},
 {'provider': 'quafu',
  'hardware_name': 'Hongluo',
  'queue_length': 0,
  'status': None,
  'is_toll': None,
  'raw': {'queue_length': 0}}]

In [8]:
bk_sim = Backend("Baihua")

print("chip:", bk_sim.chip_name)
print("two_qubit_gate_basis:", bk_sim.two_qubit_gate_basis)
print("n nodes:", bk_sim.graph.number_of_nodes())
print("n edges:", bk_sim.graph.number_of_edges())

chip: Baihua
two_qubit_gate_basis: cz
n nodes: 156
n edges: 100


## 2) 低保真边过滤

用 `edge_filtered_graph(thres=...)` 观察不同阈值对拓扑连通性的影响。

In [9]:
section("Filtered graph")
for thres in [0.6, 0.9, 0.99]:
    g = bk_sim.edge_filtered_graph(thres=thres)
    print(f"thres={thres}: nodes={g.number_of_nodes()}, edges={g.number_of_edges()}")


================== Filtered graph ==================
thres=0.6: nodes=152, edges=100
thres=0.9: nodes=152, edges=100
thres=0.99: nodes=143, edges=18


## 3) 导出拓扑图

调用 `draw(save_svg_fname=..., edge_fidelity_thres=...)` 导出 SVG。

In [10]:
section("Draw topology")
out_dir = Path("examples") / "_artifacts"
out_dir.mkdir(parents=True, exist_ok=True)

svg_prefix = out_dir / "gd_qc1_topology"
bk_sim.draw(save_svg_fname=str(svg_prefix), edge_fidelity_thres=0.9)
print("Saved:", str(svg_prefix) + ".svg")


================== Draw topology ==================
Saved: examples\_artifacts\gd_qc1_topology.svg


## 4) 自定义后端（完全离线）

可用字典自定义芯片拓扑，适合本地算法调试或测试。

In [11]:
section("Custom backend")
custom_chip = {
    "size": (3, 1),
    "priority_qubits": [[0, 1, 2]],
    "qubits_info": {
        "Q0": {"fidelity": 0.99, "coordinate": [0.0, 0.0], "T1": 0.0, "T2": 0.0, "frequency": 0.0},
        "Q1": {"fidelity": 0.98, "coordinate": [1.0, 0.0], "T1": 0.0, "T2": 0.0, "frequency": 0.0},
        "Q2": {"fidelity": 0.97, "coordinate": [2.0, 0.0], "T1": 0.0, "T2": 0.0, "frequency": 0.0},
    },
    "couplers_info": {
        "C0": {"qubits_index": [0, 1], "fidelity": 0.96, "index": 0},
        "C1": {"qubits_index": [1, 2], "fidelity": 0.95, "index": 1},
    },
    "global_info": {
        "two_qubit_gate_basis": "cz",
        "nqubits_available": 3,
        "error_rate_2q": 0.04,
        "one_qubit_gate_length": 1.0,
        "two_qubit_gate_length": 1.0,
    },
    "calibration_time": "custom",
}

bk_custom = Backend(custom_chip)
print("custom nodes:", bk_custom.graph.number_of_nodes())
print("custom edges:", bk_custom.graph.number_of_edges())
print("basis:", bk_custom.two_qubit_gate_basis)


================== Custom backend ==================
custom nodes: 3
custom edges: 2
basis: cz
